In [ ]:
!nvcc --version
!pip install git+https://github.com/andreinechaev/nvcc4jupyter
%load_ext nvcc4jupyter

In [ ]:
%%cuda
#include <stdio.h>

__global__ void op1(int a[], int alen, int t_trds) {
    int tid = threadIdx.x;
    for (int i=tid;i<alen;i=i+t_trds) {
        a[i]=i*i;
    }
}

__global__ void op2(int a[], int alen, int t_trds) {
    int tid = threadIdx.x;
    for (int i=tid;i<alen;i=i+t_trds) {
        a[i]=i*i*i;
    }
}

__global__ void op3(int a[], int b[], int alen, int t_trds) {
    int tid = threadIdx.x;
    for (int i=tid;i<alen;i=i+t_trds) {
        b[i]=a[i]+b[i];
    }
}

int main(){
    int a[3200], *da;
    int b[3200], *db;
    for(int i=0;i<3200;i++) {
        a[i]=i+1;
        b[i]=i+1;
    }
    cudaMalloc(&da, 3200*sizeof(int));
    cudaMalloc(&db, 3200*sizeof(int));
    cudaMemcpy(da,a,3200*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(db,b,3200*sizeof(int), cudaMemcpyHostToDevice);
    op1<<<1,32>>>(da, 3200, 32);
    op2<<<1,32>>>(db, 3200, 32);
    op3<<<1,32>>>(da, db, 3200, 32);
    cudaMemcpy(b,db,3200*sizeof(int), cudaMemcpyDeviceToHost);

}

In [ ]:
%%cuda
#include <stdio.h>
#include <math.h>

__global__ void eucd(int a[], int N, double m[]) {
    int tid = blockIdx.x*1024+threadIdx.x;
    int col = tid%N;
    int row = tid/N;
    if(row<N && col<N) {
        double dx = a[row*2] - a[col*2];
        double dy = a[row*2+1] - a[col*2+1];
        m[row*N+col] = sqrt((dx * dx) + (dy * dy));
    }
}

int main(){
    int N = 5000;
    int *vector, *hvector;
    double *mvector, *cvector;
    cudaMalloc(&vector, N * 2 * sizeof(int));
    cudaMalloc(&mvector, N * N * sizeof(double));
    hvector = (int*)malloc(N*2*sizeof(int));
    cvector = (double*)malloc(N*N*sizeof(double));
    int pairs = N*N;
    for(int i=0;i<N;i++) {
        hvector[i*2] = i*2;
        hvector[i*2+1] = i*2+1;
    }
    cudaMemcpy(vector,hvector,N*2*sizeof(int), cudaMemcpyHostToDevice);
    int nblocks = ceil((pairs*1.0)/(1024*1.0));
    eucd<<<nblocks,1024>>>(vector, N, mvector);
    cudaMemcpy(cvector,mvector,N*N*sizeof(double), cudaMemcpyDeviceToHost);
    double maxi = 0;
    for(int i=0;i<N*N;i++) {
        if(cvector[i]>maxi) {
            maxi=cvector[i];
        }
    }
    printf("%f", maxi);
}

In [ ]:
%%cuda --compiler-args "-rdc=true -lcudadevrt"
#include <stdio.h>
#include <math.h>

__global__ void child(int a[], double m[], int tid, int N) {
    int ttid = tid + blockIdx.x*1024+threadIdx.x;
    if(tid<N && ttid<N) {
        double dx = a[tid*2] - a[ttid*2];
        double dy = a[tid*2+1] - a[ttid*2+1];
        m[tid*N+ttid] = sqrt((dx * dx) + (dy * dy));
    }

}

__global__ void eucd(int a[], int N, double m[]) {
    int tid = blockIdx.x*1024+threadIdx.x;
    if(tid<N) {
        int rem = N-tid-1;
        int nblocks = ceil((rem*1.0)/(1024*1.0));
        child<<<nblocks,1024>>>(a, m, tid, N);
    }
}

int main(){
    int N = 5000;
    int *vector, *hvector;
    double *mvector, *cvector;
    cudaMalloc(&vector, N * 2 * sizeof(int));
    cudaMalloc(&mvector, N * N * sizeof(double));
    hvector = (int*)malloc(N*2*sizeof(int));
    cvector = (double*)malloc(N*N*sizeof(double));
    for(int i=0;i<N;i++) {
        hvector[i*2] = i*2;
        hvector[i*2+1] = i*2+1;
    }
    cudaMemcpy(vector,hvector,N*2*sizeof(int), cudaMemcpyHostToDevice);
    int nblocks = ceil((N*1.0)/(1024*1.0));
    eucd<<<nblocks,1024>>>(vector, N, mvector);
    cudaMemcpy(cvector,mvector,N*N*sizeof(double), cudaMemcpyDeviceToHost);
    double maxi = 0;
    for(int i=0;i<N*N;i++) {
        if(cvector[i]>maxi) {
            maxi=cvector[i];
        }
    }
    printf("%f", maxi);
}

In [ ]:
# This is column major access, which turns out to be slow due to absence of adjacent reads in columns

%%cuda --compiler-args "-rdc=true -lcudadevrt"
#include <stdio.h>
#include <math.h>

__global__ void matsq(int a[], int N, int m[]) {
    int tid = blockIdx.x*1024+threadIdx.x;
    int col = tid%N;
    int row = tid/N;
    if(row<N && col<N) {
        for(int i=0;i<N;i++) {
            m[row*N+col] += (a[row*N+i]*a[i*N+col]);
        }
    }
}

int main(){
    int N = 5000;
    int *vector, *hvector;
    int *mvector, *cvector;
    cudaMalloc(&vector, N * N * sizeof(int));
    cudaMalloc(&mvector, N * N * sizeof(int));
    cudaMemset(mvector, 0, N * N * sizeof(int));
    hvector = (int*)malloc(N*N*sizeof(int));
    cvector = (int*)malloc(N*N*sizeof(double));
    for(int i=0;i<N;i++) {
        for(int j=0;j<N;j++) {
            hvector[i*N+j] = i+j;
        }
    }
    cudaMemcpy(vector,hvector,N*N*sizeof(int), cudaMemcpyHostToDevice);
    int nblocks = ceil((N*N*1.0)/(1024*1.0));
    matsq<<<nblocks,1024>>>(vector, N, mvector);
    cudaMemcpy(cvector,mvector,N*N*sizeof(int), cudaMemcpyDeviceToHost);
}

In [ ]:
# Find the maximum in a large array parallely code:
%%cuda --compiler-args "-rdc=true -lcudadevrt"
#include <stdio.h>
#include <math.h>
#include <stdlib.h>
#include <assert.h>

__global__ void maxf(int a[], int N, int step) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int maxi = 0;
    for(int i=tid;i<N;i=i+step) {
        if(a[i]>maxi) {
            maxi=a[i];
        }
    }
    a[tid] = maxi;
}

int main(){
    int N = 500000;
    int k = 100;
    int *vector;
    vector = (int*)malloc(N*sizeof(int));
    int *mvector;
    cudaMalloc(&mvector, N * sizeof(int));
    int maxi = 0;
    for(int i=0;i<N;i++) {
        vector[i] = rand();
        if(vector[i]>maxi) {
            maxi=vector[i];
        }
    }
    cudaMemcpy(mvector,vector,N*sizeof(int), cudaMemcpyHostToDevice);
    int reqd_t = ceil((N*1.0)/(k*1.0));
    int nblocks = ceil((reqd_t*1.0)/(1024*1.0));
    int xtra_t = ceil((nblocks*1024-reqd_t)*1.0/(nblocks*1.0));
    int step = nblocks*(1024-xtra_t);
    maxf<<<nblocks,1024-xtra_t>>>(mvector, N, step);
    cudaMemcpy(vector,mvector,N*sizeof(int), cudaMemcpyDeviceToHost);
    for(int i=1;i<step;i++) {
       vector[i*k] = vector[i]; // Give as many threads so that i*k does not overflow
    }
    cudaMemcpy(mvector,vector,N*sizeof(int), cudaMemcpyHostToDevice);
    nblocks = ceil((k*1.0)/(1024*1.0));
    xtra_t = ceil((nblocks*1024-k)*1.0/(nblocks*1.0));
    step = nblocks*(1024-xtra_t);
    maxf<<<nblocks,1024-xtra_t>>>(mvector, N, step);
    cudaMemcpy(vector,mvector,N*sizeof(int), cudaMemcpyDeviceToHost);
    assert(maxi==vector[0]);
}

In [ ]:
# Write kernels to encrypt and decrypt messages.
# Assume that the message contains only a..z.
# – Encrypt: each character c becomes c+1. z becomes a.
# – Encrypt: each ith character c becomes c+i.

# Find the maximum in a large array parallely code:
%%cuda --compiler-args "-rdc=true -lcudadevrt"
#include <stdio.h>
#include <math.h>
#include <stdlib.h>
#include <assert.h>

__global__ void en_1(char a[]) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int is_z = (a[tid]=='z');
    a[tid] = a[tid] + 1 - is_z*26;
}

__global__ void dec_1(char a[]) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int is_a = (a[tid]=='a');
    a[tid] = a[tid] - 1 + is_a*26;
}

__global__ void en_2(char a[]) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int offset = tid%26;
    int is_more = ('z'- a[tid]) > offset;
    int pos_off = a[tid] - 'a';
    a[tid] = a[tid] - is_more*pos_off;
    offset = offset - ('z'- a[tid]) - 1;
    a[tid] = a[tid] + offset;
}

__global__ void dec_2(char a[]) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int offset = tid%26;
    int is_more = (a[tid] -'a') > offset;
    int pos_off = 'z' - a[tid];
    a[tid] = a[tid] + is_more*pos_off;
    offset = offset - (a[tid] -'a') - 1;
    a[tid] = a[tid] - offset;
}

int main(){
    int N = 15;
    char *vector, *avector, *cvector;
    vector = (char*)malloc(N * sizeof(char));
    avector = (char*)malloc(N * sizeof(char));
    cvector = (char*)malloc(N * sizeof(char));
    char *mvector;
    cudaMalloc(&mvector, N * sizeof(char));
    for(int i=0;i<N;i++) {
        vector[i] = 'a' + (rand() % 26);
    }
    printf("%s ", vector);
    cudaMemcpy(mvector,vector,N*sizeof(char), cudaMemcpyHostToDevice);
    en_1<<<1,15>>>(mvector);
    dec_1<<<1,15>>>(mvector);
    cudaMemcpy(avector,mvector,N*sizeof(char), cudaMemcpyDeviceToHost);
    printf("%s ", avector);

    cudaMemcpy(mvector,vector,N*sizeof(char), cudaMemcpyHostToDevice);
    en_2<<<1,15>>>(mvector);
    dec_2<<<1,15>>>(mvector);
    cudaMemcpy(cvector,mvector,N*sizeof(char), cudaMemcpyDeviceToHost);
    printf("%s ", avector);
}


In [ ]:
# ● Parallelize run-length-encoding to compress data.
# – e.g., if input is 0001101000100011110111010001 then the
# output is 032113134131131. The initial bit is same as
# input, followed by frequencies of that bit and its negation.
# – For the same input, another compression output is
# 4271111154213261301. This stores index and frequency.
# We cannot use heap as multiple threads would insert to the heap & it would be a race condition

# Find the maximum in a large array parallely code:
%%cuda --compiler-args "-rdc=true -lcudadevrt"
#include <stdio.h>
#include <math.h>
#include <stdlib.h>
#include <string.h>

__global__ void en_1(char a[], int N, int hvector[]) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int is_strt = (tid!=0);
    int is_elig = (a[tid]!=a[tid-1*is_strt]) || !is_strt;
    int count=1;
    int is_eq=1;
    for(int i=tid+1;i<N*is_elig*is_eq;i++) {
        is_eq = (a[i]==a[i-1]);
        count = count + is_eq;
    }
    hvector[tid*2] = tid;
    hvector[tid*2+1] = count*is_elig;

}

int main(){
    int N=29;
    int *hvector, *avector;
    avector = (int*)malloc(N*2*sizeof(int));
    cudaMalloc(&hvector, N * 2 * sizeof(int));
    cudaMemset(hvector, 0xFF, N * 2 * sizeof(int));
    char *vector;
    vector = (char*)malloc(N * sizeof(char));
    char *mvector;
    cudaMalloc(&mvector, N * sizeof(char));
    strcpy(vector, "0001101000100011110111010001");
    cudaMemcpy(mvector,vector,N*sizeof(char), cudaMemcpyHostToDevice);
    en_1<<<1,N>>>(mvector, N, hvector);
    cudaMemcpy(avector,hvector,N*2*sizeof(int), cudaMemcpyDeviceToHost);
    printf("%d", vector[0]-'0');
    for(int i=0;i<N;i++) {
        if(avector[i*2+1]!=0) {
            printf("%d", avector[i*2+1]);
        }
    }
    printf("\n");
    for(int i=0;i<N;i++) {
        if(vector[i]=='1' && avector[i*2+1]!=0) {
            printf("%d%d", i+1,avector[i*2+1]);
        }
    }
}

In [ ]:
# ● Parallelize run-length-encoding to compress data.
# – e.g., if input is 0001101000100011110111010001 then the
# output is 032113134131131. The initial bit is same as
# input, followed by frequencies of that bit and its negation.
# – For the same input, another compression output is
# 4271111154213261301. This stores index and frequency.
# We cannot use heap as multiple threads would insert to the heap & it would be a race condition
# Version 2 using prefix sum

# Find the maximum in a large array parallely code:
%%cuda --compiler-args "-rdc=true -lcudadevrt"
#include <stdio.h>
#include <math.h>
#include <stdlib.h>
#include <string.h>

__global__ void start_idx(char a[], int hvector[]) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int is_strt = (tid!=0);
    int is_elig = (a[tid]!=a[tid-1*is_strt]) || !is_strt;
    hvector[tid]=is_elig;
}
__global__ void all_start(int h[], int b[]) {
    int tid = blockIdx.x*blockDim.x+threadIdx.x;
    int is_strt = (tid!=0);
    int is_elig = (h[tid]!=h[tid-1*is_strt]) || !is_strt;
    b[h[tid]-1]=(tid+1)*is_elig;
}

int main(){
    int N=29;
    int *hvector, *avector, *bvector, *cvector;
    avector = (int*)malloc((N-1)*sizeof(int));
    cvector = (int*)malloc((N-1)*sizeof(int));
    cudaMalloc(&hvector, (N-1) * sizeof(int));
    cudaMalloc(&bvector, (N-1) * sizeof(int));
    char *vector;
    vector = (char*)malloc(N * sizeof(char));
    char *mvector;
    cudaMalloc(&mvector, N * sizeof(char));
    strcpy(vector, "0001101000100011110111010001");
    cudaMemcpy(mvector,vector,N*sizeof(char), cudaMemcpyHostToDevice);
    start_idx<<<1,N-1>>>(mvector,hvector);
    cudaMemcpy(avector,hvector,(N-1)*sizeof(int), cudaMemcpyDeviceToHost);
    for(int i=1;i<N-1;i++) {
        avector[i] += avector[i-1];
    }
    cudaMemcpy(hvector,avector,(N-1)*sizeof(int), cudaMemcpyHostToDevice);
    all_start<<<1,N-1>>>(hvector,bvector);
    cudaMemcpy(cvector,bvector,(N-1)*sizeof(int), cudaMemcpyDeviceToHost);
    printf("%d", vector[0]-'0');
    int idx = -1;
    for(int i=1;i<N-1;i++) {
        if(cvector[i]==0) {
            idx = i-1;
            break;
        }
        printf("%d", cvector[i]-cvector[i-1]);
    }
    printf("%d", N-cvector[idx]);
    printf("\n");
    idx=!(vector[0]-'0');
    for(int i=idx;i<N-1;i=i+2) {
        if(cvector[i]==0) {
            break;
        }
        if(cvector[i+1]!=0) {
            printf("%d%d", cvector[i],cvector[i+1]-cvector[i]);
        }
        else {
            printf("%d%d", cvector[i],N-cvector[i]);
        }
    }
}